# Qt–Ft agglomeration: analysis

Works on finished runs in `runs/single/` and `runs/ensemble/`. Nothing here changes the simulation output;
results are written to `<run>/analysis/`.

In [ ]:
%matplotlib inline
%load_ext autoreload
# re-import changed qtft code before every cell; after larger code changes, restart the kernel
%autoreload 2
from pathlib import Path
import pandas as pd
from qtft import data_path, ensemble_stats, load_stats, compare, update_index

DATA = data_path()               # <repo>/runs

## 1. All runs

In [ ]:
update_index(DATA)              # one line per run: status, key parameters, what differs from the defaults

## 2. Ensemble statistics of one run

In [ ]:
run_dir = DATA / "single" / ""   # group and name from the index above

stats = ensemble_stats(run_dir) # mean ± SD over finished replicas; saved to <run>/analysis/
stats["final"]                  # final-state values (mean, SD, number of replicas)

In [ ]:
stats["per_replica"]            # half-times, percolation and final values per replica

## 3. Full report of one run

Ensemble statistics, structure analysis with OVITO (Rg, compactness, cluster centres, nearest-neighbour
distances, composition; every `every`-th trajectory frame), overlap diagnostics, the binding rate k_a
(Collins–Kimball), all figures (PDF + SVG + PNG in `figures/<figure>/`, each panel also alone there), the
final-state table (CSV + LaTeX) and `report.pdf`: cover pages (run, simulation parameters, final state) and all
figures, one per page, to review them at a glance. Everything goes to `<run>/analysis/`. The figures, in report
order: calcium, overview, overlap, cluster sizes, agglomerates above size, agglomerate distributions, FIB-SEM
comparison, binding rate. A comparison (section 5) and a sweep (section 6) show the same figures for several runs.

In [ ]:
from qtft import make_report
out = make_report(run_dir, every=10,     # every=1: all frames (slower)
                  n_min=[5, 20, 60, 80, 100, 150],      # agglomerates above size: >= n particles
                  n_min_qt=[5, 10, 20, 50, 70],         # >= n Qt
                  n_min_ft=[5, 20, 60, 80, 100, 150],   # >= n Ft
                  n_qt_ft=(15, 20))                     # last row: >= 15 Qt and >= 20 Ft together
pd.read_csv(out / "summary_table.csv", index_col=0)

In [ ]:
from IPython.display import Image, display
# saved by make_report, shown once each; width in pixels in proportion to the figure width (183 or 120 mm)
for name, width in (("overview", 900), ("agglomerate_distributions", 900), ("agglomerates_above_size", 590)):
    display(Image(filename=out / "figures" / name / f"{name}.png", width=width))

In [ ]:
from qtft.figures import load_run, fig_calcium

# calcium protocol: calcium and loading f, rate constants, binding and unbinding rates, apparent k_a,
# bonds and broken bonds, fraction bound (dotted lines: influx start and end); cluster sizes are in cluster_sizes
fig_calcium(load_run(run_dir));

In [ ]:
from qtft import Params, binding_rate
from qtft.figures import load_run, fig_binding_rate

# association rate k_a (Collins–Kimball contact reaction) from the start of the bond curve, with the theory;
# saved to <run>/analysis/binding_rate.csv, figure in analysis/figures/binding_rate/
# needs a calcium step (ca_rise_s = 0): the fit assumes the peak rate from the influx start on
if Params.load(run_dir / "params.json").ca_rise_s > 0:
    print("calcium rises gradually (ca_rise_s > 0): binding rate not measured")
else:
    rate = binding_rate(run_dir)
    fig_binding_rate(load_run(run_dir), rate=rate)
    display(rate["summary"])

Individual figures can also be drawn and adjusted interactively:

```python
from qtft import bond_rates
from qtft.figures import (load_run, fig_cluster_sizes, fig_agglomerates_above_size_total,
                         fig_agglomerates_above_size, fig_bond_rates)
run = load_run(run_dir)
fig_cluster_sizes([run], traces=True)   # mean and weight-average sizes, with individual replicas
fig_agglomerates_above_size_total([run], n_min=[5, 20], density=True)   # >= 5 and >= 20 particles per 1000 µm³
fig_agglomerates_above_size(run, n_min_qt=[5, 10, 20]) # >= n particles / Qt / Ft, number and density
fig_bond_rates(run, rates=bond_rates(run_dir, bin_s=1.0, save=False))   # the two rate panels, 1 s bins
```

Every figure of the report also takes a list of runs (one colour per run), for example
`fig_agglomerates_above_size(load_runs([run_a, run_b], ["A", "B"]))` with `load_runs` from `qtft.figures`.

In [ ]:
pd.read_csv(out / "overlap_final.csv", index_col=0)      # interpenetration per pair family (last frames)

## 4. Comparison with 3D FIB-SEM

FIB-SEM resolves the encapsulins (Qt) but not ferritin, so these metrics use the Qt only. They are computed by
`make_report` (section 3, cached in `<run>/analysis/qt_*.csv`) for the run `run_dir`, pooled over replicas (a
comparison or sweep shows them for several runs, one line per run at the end of the influx):

- **Nearest Qt in the agglomerate**: for every Qt in an agglomerate with ≥ 2 Qt, the distance to the nearest Qt
  of the same agglomerate, at the end of the influx; the peak is marked.
- **Pair correlation g(r)** of all Qt at the end of the influx; the first peak is marked.
- **Clark–Evans index R** of all Qt, first frame vs end of the influx: mean nearest-neighbour distance divided by
  its value for randomly placed points of the same density, 0.554 ρ^(−1/3). R = 1 random, R < 1 clustered.
- **Ripley's L(r) − r** of all Qt, first frame vs end of the influx: L = (3K/(4π))^(1/3), K(r) = mean number of
  other Qt within r divided by the Qt density. 0 random, > 0 clustered at scale r; the maximum is marked.

In [ ]:
from IPython.display import Image, display
from qtft import qt_spatial_run
spatial = qt_spatial_run(run_dir)                 # read from the cache written by make_report
display(Image(filename=out / "figures" / "fibsem_comparison" / "fibsem_comparison.png", width=900))
spatial["clark_evans"]                            # R per replica: first frame and end of the influx

## 5. Compare runs

Skipped on *Run All*: to run a cell, remove its first line `%%script false --no-raise-error`.

In [ ]:
%%script false --no-raise-error
from qtft import compare_report
runs = [DATA / "ensemble" / "<run A>", DATA / "ensemble" / "<run B>"]
compare(runs)                                   # table, columns labelled by the parameters that differ


In [ ]:
%%script false --no-raise-error
compare_report(runs, DATA / "comparisons" / "<name>")   # the report figures for all runs, table (CSV, LaTeX), report.pdf

## 6. Parameter sweep

Skipped on *Run All*: to run a cell, remove its first line `%%script false --no-raise-error`.

Results of a sweep built in `01_simulate_hek.ipynb` (section 4): one row per variant with the swept parameters and
mean ± SD over the finished replicas of the final-state values, peaks, half-times and peak cluster sizes (saved to
`<sweep>/sweep_results.csv`). `sweep_report` writes to `<sweep>/analysis/`: one plot per result against the first
swept parameter (one line per value of the others; default: peak bonds, peak largest cluster, half-times of binding
and of deagglomeration, peak weight-average size (bound particles) and peak weight-average size / N), the
side-by-side comparison of all runs (the figures of a single-run report in the same order, one colour per run, and
the table) and `report.pdf` with all of it. Variants without the structure or overlap analysis, the agglomerate
volumes, bond rates or Qt spatial statistics get them first (OVITO, a few minutes per variant, then cached).
"Final" values are those at the end of the whole protocol, after the pump-out.

In [ ]:
%%script false --no-raise-error
from qtft import sweep_results, sweep_report, sweep_runs

sweep = DATA / "sweeps" / "<name>"
results = sweep_results(sweep)            # table, also saved to <sweep>/sweep_results.csv
results.round(3)

In [ ]:
%%script false --no-raise-error
sweep_report(sweep)                       # sweep plots, comparison figures, table (CSV, LaTeX), report.pdf

In [ ]:
%%script false --no-raise-error
compare(sweep_runs(sweep))                # side-by-side final-state table

## 7. FIB-SEM export

Skipped on *Run All*: to run a cell, remove its first line `%%script false --no-raise-error`.

Frame at the end of the influx of every replica in the schema of the FIB-SEM segmentation pipeline:
encapsulins only, cluster IDs from the bond graph, clusters unwrapped across the periodic box, percolating
clusters flagged. All replicas must be finished and pass the checks, otherwise nothing is written.
Written to `<run>/rep_XX/fibsem/` (CSV, metadata JSON, x–y check plot) and pooled to `<run>/fibsem/`.

In [ ]:
%%script false --no-raise-error
from qtft import fibsem_export_run
out = fibsem_export_run(run_dir, voxel_nm=4.0)                 # <run>/fibsem/: all replicas pooled
pd.read_csv(out / "sim_encapsulin_centroids_clustered.csv", keep_default_na=False).head()   # spans "" = none